# Fire Prediction Data Ingestion - Google Earth Engine Only

This notebook uses **exclusively Google Earth Engine (GEE)** for all data sources:
- **Fire Detection**: MODIS Terra + Aqua via GEE
- **Weather Data**: ERA5-Land reanalysis via GEE
- **Terrain Features**: SRTM DEM via GEE
- **Water Distance**: JRC Global Surface Water via GEE
- **Vegetation/Fuel**: MODIS Land Cover + LANDFIRE via GEE

**Benefits of GEE-only approach:**
- No API rate limits
- Perfect temporal alignment (all data from same source)
- Historical data coverage (2000-present)
- Better data quality and consistency
- Faster processing (no external API delays)

## Pipeline Overview
1. Initialize Google Earth Engine
2. Fetch MODIS fire detections (Terra + Aqua combined)
3. Fetch ERA5 weather data for each fire location
4. Fetch terrain features (elevation, slope, ruggedness, etc.)
5. Fetch distance to water bodies
6. Fetch vegetation and fuel layer features
7. Combine all features and save to parquet

In [1]:
# ============================================================================
# IMPORTS
# ============================================================================

# Core data manipulation
import pandas as pd
import numpy as np
import geopandas as gpd

# Date and time handling
from datetime import datetime, timedelta

# File and path handling
import os
import json
from pathlib import Path

# Utilities
import warnings
import time
from typing import Dict, List, Optional, Tuple

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# ============================================================================
# DIRECTORY DEFINITIONS
# ============================================================================

# Define data directories
RAW_DIR = Path('data/raw')
PROCESSED_DIR = Path('data/processed')
MODEL_DIR = Path('models')

# Create directories if they don't exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("✓ All imports loaded successfully")
print(f"✓ Data directories initialized:")
print(f"  - Raw: {RAW_DIR}")
print(f"  - Processed: {PROCESSED_DIR}")
print(f"  - Models: {MODEL_DIR}")

✓ All imports loaded successfully
✓ Data directories initialized:
  - Raw: data\raw
  - Processed: data\processed
  - Models: models


## Configuration Parameters

Configure all dataset sources and sampling parameters below.


In [ ]:
# ============================================================================
# DATASET CONFIGURATION - GEE ONLY
# ============================================================================

# --- Sample Size Configuration ---
SAMPLE_SIZE = 3000  # Number of fire samples to process
RANDOM_STATE = 42   # Random seed for reproducibility

# --- MODIS Fire Detection Configuration (GEE) ---
# Extended date range for more fire samples (MODIS data available from 2000)
GEE_FIRE_DATE_START = '2020-01-01'  # Start date for fire detection (extended from 2023)
GEE_FIRE_DATE_END = '2023-12-31'    # End date for fire detection
GEE_FIRE_MIN_CONFIDENCE = 8  # FireMask >= 8 (nominal + high confidence, scale 7-9)
GEE_FIRE_MIN_FRP = 0  # Minimum Fire Radiative Power (MW)
GEE_FIRE_MAX_FRP = None  # Maximum Fire Radiative Power (MW) - None = no limit. Set to filter out very large fires (e.g., 100)

# --- Fire Filtering (to focus on ignitions) ---
FILTER_FIRST_DETECTIONS = True  # Only keep fires that are first-time detections at their location (ignitions)
FIRST_DETECTION_BUFFER_KM = 1.0  # Distance buffer (km) to consider a fire at the same location
FIRST_DETECTION_LOOKBACK_DAYS = 7  # Check if fire appeared in previous N days at this location

# Process year-by-year to avoid GEE timeouts and be more efficient
PROCESS_BY_YEAR = True  # Process one year at a time
YEARS_TO_PROCESS = ['2020', '2021', '2022', '2023']  # Years to process

# --- ERA5 Weather Configuration (GEE) ---
GEE_WEATHER_PRODUCT = 'ECMWF/ERA5_LAND/DAILY_AGGR'  # ERA5-Land daily aggregated
GEE_WEATHER_BANDS = [
    'temperature_2m',           # 2m temperature (K)
    'temperature_2m_max',       # Daily max temperature
    'temperature_2m_min',       # Daily min temperature
    'dewpoint_temperature_2m',  # Dewpoint (K) - for humidity
    'total_precipitation_sum',  # Total precipitation (m)
    'u_component_of_wind_10m',  # U wind component (m/s)
    'v_component_of_wind_10m',  # V wind component (m/s)
    'surface_pressure'          # Surface pressure (Pa)
]
WEATHER_LOOKBACK_DAYS = 14  # Days before fire event to fetch weather data

# --- Google Earth Engine Configuration ---
GEE_PROJECT = 'ee-fireprediction'  # Your GEE project ID (or use environment variable)
GEE_KEY_PATH = None  # Will be loaded from environment variable 'GEE_KEY'
GEE_SERVICE_ACCOUNT = None  # Will be loaded from GEE credentials JSON
GEE_USE_SERVICE_ACCOUNT = True  # Use service account (True) or user auth (False)

# --- Feature Flags ---
FETCH_WEATHER = True
FETCH_TERRAIN = True
FETCH_WATER_DISTANCE = True
FETCH_VEGETATION = True

# --- Terrain Configuration ---
TERRAIN_SCALE_METERS = 30.0  # Resolution for terrain extraction
TERRAIN_BUFFER_KM = 1.0  # Buffer radius around point for terrain stats

# --- Geographic Filtering ---
GEOGRAPHIC_BOUNDS = None  # Set to None to process all locations, or [min_lon, min_lat, max_lon, max_lat]

# --- Stratification (for sampling) ---
ENABLE_GEOGRAPHIC_STRATIFICATION = True  # Ensure diversity across lat/lon grid
N_LAT_BINS = 6   # Number of latitude bins
N_LON_BINS = 12  # Number of longitude bins (72 total regions)

# --- Negative Sample Generation ---
GENERATE_NEGATIVE_SAMPLES = True  # Generate non-fire locations for binary classification
N_NEGATIVE_SAMPLES = SAMPLE_SIZE  # Match number of positive samples
NEGATIVE_SAMPLE_STRATEGY = 'spatial_temporal'  # Options: 'spatial_temporal', 'random', 'grid'

# Print configuration
print("="*80)
print("FIRE PREDICTION DATA INGESTION - GEE ONLY CONFIGURATION")
print("="*80)
print(f"\n📊 Dataset Configuration:")
print(f"   Sample Size: {SAMPLE_SIZE}")
print(f"   Random State: {RANDOM_STATE}")
print(f"\n🔥 MODIS Fire Data (GEE):")
print(f"   Date Range: {GEE_FIRE_DATE_START} to {GEE_FIRE_DATE_END}")
print(f"   Process by Year: {PROCESS_BY_YEAR}")
if PROCESS_BY_YEAR:
    print(f"   Years: {', '.join(YEARS_TO_PROCESS)}")
print(f"   Min Confidence: FireMask >= {GEE_FIRE_MIN_CONFIDENCE}")
print(f"   Min FRP: {GEE_FIRE_MIN_FRP} MW")
if GEE_FIRE_MAX_FRP is not None:
    print(f"   Max FRP: {GEE_FIRE_MAX_FRP} MW (filtering large fires)")
print(f"   Filter First Detections: {FILTER_FIRST_DETECTIONS}")
if FILTER_FIRST_DETECTIONS:
    print(f"     Buffer: {FIRST_DETECTION_BUFFER_KM} km, Lookback: {FIRST_DETECTION_LOOKBACK_DAYS} days")
print(f"\n📉 Negative Samples:")
print(f"   Generate: {GENERATE_NEGATIVE_SAMPLES}")
if GENERATE_NEGATIVE_SAMPLES:
    print(f"   Strategy: {NEGATIVE_SAMPLE_STRATEGY}")
    print(f"   Count: {N_NEGATIVE_SAMPLES}")
print(f"\n🌤️  Weather Data (GEE ERA5):")
print(f"   Product: {GEE_WEATHER_PRODUCT}")
print(f"   Variables: {len(GEE_WEATHER_BANDS)}")
print(f"   Lookback: {WEATHER_LOOKBACK_DAYS} days")
print(f"\n🌍 Geospatial Features (GEE):")
print(f"   Terrain: {FETCH_TERRAIN}")
print(f"   Water Distance: {FETCH_WATER_DISTANCE}")
print(f"   Vegetation: {FETCH_VEGETATION}")
print(f"\n📍 Stratification:")
print(f"   Geographic: {ENABLE_GEOGRAPHIC_STRATIFICATION} ({N_LAT_BINS}x{N_LON_BINS} grid = {N_LAT_BINS*N_LON_BINS} regions)")
print("="*80)


FIRE PREDICTION DATA INGESTION - GEE ONLY CONFIGURATION

📊 Dataset Configuration:
   Sample Size: 3000
   Random State: 42

🔥 MODIS Fire Data (GEE):
   Date Range: 2020-01-01 to 2023-12-31
   Process by Year: True
   Years: 2020, 2021, 2022, 2023
   Min Confidence: FireMask >= 8
   Min FRP: 0 MW

📉 Negative Samples:
   Generate: True
   Strategy: spatial_temporal
   Count: 3000

🌤️  Weather Data (GEE ERA5):
   Product: ECMWF/ERA5_LAND/DAILY_AGGR
   Variables: 8
   Lookback: 14 days

🌍 Geospatial Features (GEE):
   Terrain: True
   Water Distance: True
   Vegetation: True

📍 Stratification:
   Geographic: True (6x12 grid = 72 regions)


## Step 1: Initialize Google Earth Engine


In [3]:
# ============================================================================
# GOOGLE EARTH ENGINE INITIALIZATION
# ============================================================================

import ee

print("="*80)
print("INITIALIZING GOOGLE EARTH ENGINE")
print("="*80)

try:
    # Get project ID
    project = os.getenv('GEE_PROJECT', GEE_PROJECT)
    
    if GEE_USE_SERVICE_ACCOUNT:
        # Service Account Authentication (recommended for automated workflows)
        print("\nUsing Service Account authentication...")
        
        # Get credentials path from environment variable
        gee_key_path = os.getenv('GEE_KEY', GEE_KEY_PATH)
        
        if not gee_key_path:
            raise ValueError("GEE_KEY environment variable not set. Please set it to your service account JSON path.")
        
        # Clean up path (handle quotes and special characters)
        gee_key_path = gee_key_path.strip('"\'')
        gee_key_path = ''.join(c for c in gee_key_path if ord(c) >= 32 or c in '\\\\/')
        gee_key_path = gee_key_path.replace('/', '\\\\')
        gee_key_path = gee_key_path.strip()
        
        # Check if file exists
        if not os.path.isfile(gee_key_path):
            # Try to find JSON file in current directory
            print(f"  Credentials not found at {gee_key_path}")
            print(f"  Searching in project directory...")
            for jf in Path('.').glob('*.json'):
                if any(keyword in jf.name.lower() for keyword in ['service', 'gee', 'fire', 'earth']):
                    gee_key_path = str(jf.resolve())
                    print(f"  Found credentials: {gee_key_path}")
                    break
        
        if not os.path.isfile(gee_key_path):
            raise FileNotFoundError(f"Cannot find GEE credentials file: {gee_key_path}")
        
        print(f"✓ Using credentials: {gee_key_path}")
        
        # Read service account email from JSON
        with open(gee_key_path, 'r') as f:
            key_data = json.load(f)
            service_account_email = key_data.get('client_email')
            if not service_account_email:
                raise ValueError("Service account JSON missing 'client_email' field")
        
        print(f"✓ Service account: {service_account_email}")
        
        # Initialize with service account
        credentials = ee.ServiceAccountCredentials(service_account_email, gee_key_path)
        ee.Initialize(credentials)
        
    else:
        # User Authentication (interactive)
        print("\nUsing User authentication...")
        print("If this is your first time, you'll need to authenticate in your browser.")
        
        try:
            # Try to initialize (will use cached credentials if available)
            ee.Initialize(project=project)
            print(f"✓ Initialized with cached credentials")
        except Exception as e:
            # Need to authenticate
            print(f"✓ Authentication required...")
            ee.Authenticate()
            ee.Initialize(project=project)
            print(f"✓ Authentication successful")
        
        print(f"✓ Project: {project}")
    
    # Test the connection
    test = ee.Number(1).getInfo()
    print(f"\n✅ Google Earth Engine initialized successfully!")
    print(f"   Ready for fire detection, weather, terrain, and geospatial data extraction.")
    
except Exception as e:
    print(f"\n❌ Error initializing Google Earth Engine: {e}")
    print(f"\nTroubleshooting:")
    print(f"  1. Make sure you have earthengine-api installed: pip install earthengine-api")
    print(f"  2. For service account: Set GEE_KEY environment variable to your JSON key path")
    print(f"  3. For user auth: Run ee.Authenticate() first or set GEE_USE_SERVICE_ACCOUNT=False")
    print(f"  4. Make sure your GEE project ID is correct: {project}")
    raise

print("="*80)


INITIALIZING GOOGLE EARTH ENGINE

Using Service Account authentication...
✓ Using credentials: C:\Users\Drewo\OneDrive\Documents\GIT\fire_prediction\fireprediction-483622-3e2ab1191a16.json
✓ Service account: acount-1@fireprediction-483622.iam.gserviceaccount.com

✅ Google Earth Engine initialized successfully!
   Ready for fire detection, weather, terrain, and geospatial data extraction.


## Step 2: Fetch MODIS Fire Detections from GEE


In [ ]:
# ============================================================================
# FETCH MODIS FIRE DETECTIONS FROM GEE
# ============================================================================

print("="*80)
print("FETCHING MODIS FIRE DETECTIONS FROM GOOGLE EARTH ENGINE")
print("="*80)

print(f"\nConfiguration:")
print(f"  Products: MODIS/006/MOD14A1 (Terra) + MODIS/006/MYD14A1 (Aqua) - Combined")
print(f"  Date Range: {GEE_FIRE_DATE_START} to {GEE_FIRE_DATE_END}")
print(f"  Min Confidence: FireMask >= {GEE_FIRE_MIN_CONFIDENCE} (nominal + high confidence)")
print(f"  Target Samples: {SAMPLE_SIZE}")
print(f"  Extraction Method: reduceToVectors (extracts ALL fire pixels, not random sample)")

# Load MODIS fire product - combine Terra and Aqua for better coverage
fires_terra = ee.ImageCollection('MODIS/006/MOD14A1') \
    .filterDate(GEE_FIRE_DATE_START, GEE_FIRE_DATE_END)
fires_aqua = ee.ImageCollection('MODIS/006/MYD14A1') \
    .filterDate(GEE_FIRE_DATE_START, GEE_FIRE_DATE_END)
fires = fires_terra.merge(fires_aqua)

# Function to extract fire pixels as features
def extract_fire_points(image):
    """Extract fire pixels from a single MODIS image."""
    fire_mask = image.select('FireMask')
    max_frp = image.select('MaxFRP')
    
    # Filter for high confidence fires (FireMask >= 8: nominal or high confidence)
    high_confidence = fire_mask.gte(GEE_FIRE_MIN_CONFIDENCE)
    masked = image.updateMask(high_confidence)
    
    # Extract all fire pixels as vector features
    fire_vectors = masked.select(['FireMask']).reduceToVectors(
        geometry=ee.Geometry.Rectangle([-180, -90, 180, 90], None, False),
        scale=2000,  # 2km scale to avoid maxPixels error
        geometryType='centroid',
        maxPixels=1e9,
        bestEffort=True,
        crs='EPSG:4326'
    )
    
    # Add date and properties to each feature
    def add_properties(feature):
        return feature.set({
            'ACQ_DATE': image.date().format('YYYY-MM-dd'),
            'FireMask': feature.get('FireMask')
        })
    
    return fire_vectors.map(add_properties)

# Process year-by-year to be more efficient and avoid GEE timeouts
fire_list = []
target_fires = min(SAMPLE_SIZE * 3, 4000)  # Limit to stay under 5000
MAX_FIRES_PER_IMAGE = 200  # Conservative limit per image

if PROCESS_BY_YEAR:
    print("\n📅 Processing year-by-year for efficiency...")
    print("Note: Processing images individually to avoid GEE 5000 element limit")
    
    for year in YEARS_TO_PROCESS:
        if len(fire_list) >= target_fires:
            print(f"✓ Reached target of {target_fires} fires, stopping early")
            break
        
        year_start = f'{year}-01-01'
        year_end = f'{year}-12-31'
        
        print(f"\n📅 Processing year {year} ({year_start} to {year_end})...")
        
        # Load MODIS for this year
        fires_terra = ee.ImageCollection('MODIS/006/MOD14A1') \
            .filterDate(year_start, year_end)
        fires_aqua = ee.ImageCollection('MODIS/006/MYD14A1') \
            .filterDate(year_start, year_end)
        fires = fires_terra.merge(fires_aqua)
        
        collection_size = fires.size().getInfo()
        print(f"  Found {collection_size} images for {year}")
        
        if collection_size == 0:
            print(f"  ⚠ No images found for {year}, skipping...")
            continue
        
        # Limit number of images to process per year
        MAX_IMAGES_PER_YEAR = min(collection_size, 50)  # Process up to 50 images per year
        print(f"  Processing first {MAX_IMAGES_PER_YEAR} of {collection_size} images...")
        
        # Process images one at a time for this year
        for img_idx in range(MAX_IMAGES_PER_YEAR):
            if len(fire_list) >= target_fires:
                break
            
            if img_idx % 10 == 0:
                print(f"    Processing image {img_idx+1}/{MAX_IMAGES_PER_YEAR}... (collected {len(fire_list)} fires so far)")
            
            try:
                # Get single image from collection
                image = ee.Image(fires.toList(MAX_IMAGES_PER_YEAR).get(img_idx))
                
                # Extract fire points from this image
                fire_vectors = extract_fire_points(image)
                
                # Limit to avoid 5000 limit
                fire_sample = fire_vectors.limit(MAX_FIRES_PER_IMAGE)
                
                # Get features
                batch_features = fire_sample.getInfo()['features']
                fire_list.extend(batch_features)
                
            except Exception as e:
                if "5000" in str(e) or "aborted" in str(e).lower():
                    # This image has too many fires, try smaller limit
                    try:
                        fire_sample = fire_vectors.limit(100)
                        batch_features = fire_sample.getInfo()['features']
                        fire_list.extend(batch_features)
                    except:
                        # Skip this image if it still fails
                        if img_idx % 10 == 0:
                            print(f"      ⚠ Skipping image {img_idx+1} (too many fires)")
                        continue
                else:
                    # Other error, skip this image
                    if img_idx % 10 == 0:
                        print(f"      ⚠ Error on image {img_idx+1}: {str(e)[:50]}")
                    continue
        
        print(f"  ✓ Year {year} complete: {len(fire_list)} total fires collected")
else:
    # Original single-date-range processing
    print("\nProcessing fire detections...")
    print("Note: Processing images individually to avoid GEE 5000 element limit")
    
    # Load MODIS fire product - combine Terra and Aqua for better coverage
    fires_terra = ee.ImageCollection('MODIS/006/MOD14A1') \
        .filterDate(GEE_FIRE_DATE_START, GEE_FIRE_DATE_END)
    fires_aqua = ee.ImageCollection('MODIS/006/MYD14A1') \
        .filterDate(GEE_FIRE_DATE_START, GEE_FIRE_DATE_END)
    fires = fires_terra.merge(fires_aqua)
    
    # Check collection size
    collection_size = fires.size().getInfo()
    print(f"\n✓ Loaded fire collection: {collection_size} images (Terra + Aqua combined)")
    
    if collection_size == 0:
        raise ValueError(f"No images found in MODIS collections for date range {GEE_FIRE_DATE_START} to {GEE_FIRE_DATE_END}")
    
    # Limit number of images to process to avoid timeout
    MAX_IMAGES_TO_PROCESS = min(collection_size, 100)  # Process up to 100 images
    print(f"Processing first {MAX_IMAGES_TO_PROCESS} of {collection_size} images...")
    print(f"Target: {target_fires} fire detections")
    
    # Process images one at a time
    for img_idx in range(MAX_IMAGES_TO_PROCESS):
        if len(fire_list) >= target_fires:
            print(f"✓ Reached target of {target_fires} fires, stopping early")
            break
        
        if img_idx % 10 == 0:
            print(f"  Processing image {img_idx+1}/{MAX_IMAGES_TO_PROCESS}... (collected {len(fire_list)} fires so far)")
        
        try:
            # Get single image from collection
            image = ee.Image(fires.toList(MAX_IMAGES_TO_PROCESS).get(img_idx))
            
            # Extract fire points from this image
            fire_vectors = extract_fire_points(image)
            
            # Limit to avoid 5000 limit
            fire_sample = fire_vectors.limit(MAX_FIRES_PER_IMAGE)
            
            # Get features
            batch_features = fire_sample.getInfo()['features']
            fire_list.extend(batch_features)
            
        except Exception as e:
            if "5000" in str(e) or "aborted" in str(e).lower():
                # This image has too many fires, try smaller limit
                try:
                    fire_sample = fire_vectors.limit(100)
                    batch_features = fire_sample.getInfo()['features']
                    fire_list.extend(batch_features)
                except:
                    # Skip this image if it still fails
                    if img_idx % 10 == 0:
                        print(f"    ⚠ Skipping image {img_idx+1} (too many fires)")
                    continue
            else:
                # Other error, skip this image
                if img_idx % 10 == 0:
                    print(f"    ⚠ Error on image {img_idx+1}: {str(e)[:50]}")
                continue

print(f"\n✓ Retrieved {len(fire_list)} fire detections from GEE")

# Convert to DataFrame
fire_data = []
for feature in fire_list:
    props = feature.get('properties', {})
    geom = feature.get('geometry', {})
    
    # Extract coordinates
    if geom.get('type') == 'Point':
        coords = geom.get('coordinates', [0, 0])
    else:
        coords = [0, 0]
    
    # Extract ACQ_DATE
    acq_date = props.get('ACQ_DATE')
    if acq_date is None:
        acq_date = props.get('date') or props.get('system:time_start')
    
    # Extract FireMask (values 7-9: low, nominal, high confidence)
    # Convert to 0-100 scale for CONFIDENCE: 7->0, 8->50, 9->100
    fire_mask = props.get('FireMask', 0)
    if fire_mask >= 7:
        # Scale FireMask 7-9 to CONFIDENCE 0-100
        confidence = int((fire_mask - 7) * 50)  # 7->0, 8->50, 9->100
    else:
        confidence = 0
    
    fire_data.append({
        'LONGITUDE': coords[0],
        'LATITUDE': coords[1],
        'ACQ_DATE': acq_date,
        'BRIGHTNESS': props.get('MaxFRP', 0),
        'FRP': props.get('MaxFRP', 0),
        'CONFIDENCE': confidence,
        'SATELLITE': 'MODIS',
        'VERSION': '6.1',
        'DAYNIGHT': 'D',
        'is_fire': 1  # Label as positive sample (fire occurred)
    })

fire_data = pd.DataFrame(fire_data)

# Validate data
if len(fire_data) == 0:
    raise ValueError("No fire detections retrieved from GEE. Check your date range and filters.")

if 'ACQ_DATE' not in fire_data.columns:
    print("⚠ WARNING: ACQ_DATE column not found in fire data.")
    raise ValueError("ACQ_DATE column is missing from fire data")

if fire_data['ACQ_DATE'].isna().all():
    print("⚠ WARNING: All ACQ_DATE values are None.")
    raise ValueError("ACQ_DATE values are missing from all fire detections")

# Clean and convert dates
fire_data = fire_data.dropna(subset=['ACQ_DATE'])
fire_data['ACQ_DATE'] = pd.to_datetime(fire_data['ACQ_DATE'])

# ============================================================================
# FIRE FILTERING (Focus on Ignitions)
# ============================================================================
print(f"\n{'='*80}")
print("APPLYING FIRE FILTERS (Focusing on Ignitions)")
print(f"{'='*80}")

initial_count = len(fire_data)
print(f"Initial fire detections: {initial_count}")

# Filter by FRP (if max FRP is set)
if GEE_FIRE_MAX_FRP is not None:
    before_frp = len(fire_data)
    fire_data = fire_data[fire_data['FRP'] <= GEE_FIRE_MAX_FRP]
    after_frp = len(fire_data)
    print(f"✓ FRP filter (max {GEE_FIRE_MAX_FRP} MW): {before_frp} → {after_frp} fires ({before_frp - after_frp} removed)")

# Filter for first-time detections (ignitions)
if FILTER_FIRST_DETECTIONS:
    print(f"\n🔍 Filtering for first-time detections (ignitions)...")
    print(f"   Buffer: {FIRST_DETECTION_BUFFER_KM} km")
    print(f"   Lookback: {FIRST_DETECTION_LOOKBACK_DAYS} days")
    
    before_first = len(fire_data)
    
    # Sort by date to process chronologically
    fire_data = fire_data.sort_values('ACQ_DATE').reset_index(drop=True)
    
    # Calculate distance threshold in degrees (approximate)
    # 1 degree latitude ≈ 111 km, so buffer_km / 111 gives degrees
    buffer_deg = FIRST_DETECTION_BUFFER_KM / 111.0
    
    # Track which fires are first detections
    is_first_detection = []
    
    for idx, row in fire_data.iterrows():
        if idx % 500 == 0 and idx > 0:
            print(f"   Processing {idx}/{len(fire_data)}... ({sum(is_first_detection)} first detections so far)")
        
        current_date = row['ACQ_DATE']
        current_lat = row['LATITUDE']
        current_lon = row['LONGITUDE']
        
        # Look back N days from current date
        lookback_date = current_date - pd.Timedelta(days=FIRST_DETECTION_LOOKBACK_DAYS)
        
        # Find fires at similar location in the lookback period
        nearby_fires = fire_data[
            (fire_data['ACQ_DATE'] >= lookback_date) &
            (fire_data['ACQ_DATE'] < current_date) &
            (fire_data['LATITUDE'] >= current_lat - buffer_deg) &
            (fire_data['LATITUDE'] <= current_lat + buffer_deg) &
            (fire_data['LONGITUDE'] >= current_lon - buffer_deg) &
            (fire_data['LONGITUDE'] <= current_lon + buffer_deg)
        ]
        
        # If no nearby fires in lookback period, this is a first detection (ignition)
        is_first_detection.append(len(nearby_fires) == 0)
    
    fire_data['is_first_detection'] = is_first_detection
    fire_data = fire_data[fire_data['is_first_detection'] == True].drop(columns=['is_first_detection'])
    
    after_first = len(fire_data)
    print(f"✓ First detection filter: {before_first} → {after_first} fires ({before_first - after_first} removed)")
    print(f"   Kept {after_first/before_first*100:.1f}% as potential ignitions")

final_count = len(fire_data)
print(f"\n📊 Filtering Summary:")
print(f"   Initial: {initial_count} fires")
print(f"   Final: {final_count} fires")
print(f"   Removed: {initial_count - final_count} fires ({(initial_count - final_count)/initial_count*100:.1f}%)")
print(f"{'='*80}\n")

# Apply geographic bounds if specified
if GEOGRAPHIC_BOUNDS:
    min_lon, min_lat, max_lon, max_lat = GEOGRAPHIC_BOUNDS
    fire_data = fire_data[
        (fire_data['LONGITUDE'] >= min_lon) & (fire_data['LONGITUDE'] <= max_lon) &
        (fire_data['LATITUDE'] >= min_lat) & (fire_data['LATITUDE'] <= max_lat)
    ]
    print(f"✓ Filtered to geographic bounds: {len(fire_data)} detections")

# Stratified geographic sampling if enabled
if ENABLE_GEOGRAPHIC_STRATIFICATION and len(fire_data) > SAMPLE_SIZE:
    def stratified_geographic_sampling(df, n_samples, random_state=42):
        """Sample fires with geographic diversity."""
        df = df.copy()
        df['lat_bin'] = pd.cut(df['LATITUDE'], bins=N_LAT_BINS, labels=False)
        df['lon_bin'] = pd.cut(df['LONGITUDE'], bins=N_LON_BINS, labels=False)
        
        sampled = df.groupby(['lat_bin', 'lon_bin']).apply(
            lambda x: x.sample(min(len(x), max(1, n_samples // (N_LAT_BINS * N_LON_BINS))), 
                             random_state=random_state)
        ).reset_index(drop=True)
        
        if len(sampled) < n_samples:
            remaining = n_samples - len(sampled)
            extra = df[~df.index.isin(sampled.index)].sample(
                min(remaining, len(df) - len(sampled)), 
                random_state=random_state
            )
            sampled = pd.concat([sampled, extra]).reset_index(drop=True)
        else:
            sampled = sampled.sample(min(len(sampled), n_samples), random_state=random_state)
        
        return sampled.drop(['lat_bin', 'lon_bin'], axis=1)
    
    fire_data = stratified_geographic_sampling(fire_data, SAMPLE_SIZE, RANDOM_STATE)
    print(f"✓ Sampled to {len(fire_data)} detections with geographic diversity")
elif len(fire_data) > SAMPLE_SIZE:
    fire_data = fire_data.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
    print(f"✓ Randomly sampled to {len(fire_data)} detections")
else:
    print(f"✓ Using all {len(fire_data)} detections")

# Ensure ACQ_DATE is datetime
if not pd.api.types.is_datetime64_any_dtype(fire_data['ACQ_DATE']):
    fire_data['ACQ_DATE'] = pd.to_datetime(fire_data['ACQ_DATE'])

# Save raw fire detection data
output_path = RAW_DIR / 'fire_detections_gee.parquet'
fire_data.to_parquet(output_path, index=False)
print(f"\n✓ Saved raw fire detections to {output_path}")
print(f"  Total detections: {len(fire_data)}")
print(f"  Date range: {fire_data['ACQ_DATE'].min()} to {fire_data['ACQ_DATE'].max()}")
print("="*80)


FETCHING MODIS FIRE DETECTIONS FROM GOOGLE EARTH ENGINE

Configuration:
  Products: MODIS/006/MOD14A1 (Terra) + MODIS/006/MYD14A1 (Aqua) - Combined
  Date Range: 2020-01-01 to 2023-12-31
  Min Confidence: FireMask >= 8 (nominal + high confidence)
  Target Samples: 3000
  Extraction Method: reduceToVectors (extracts ALL fire pixels, not random sample)

📅 Processing year-by-year for efficiency...
Note: Processing images individually to avoid GEE 5000 element limit

📅 Processing year 2020 (2020-01-01 to 2020-12-31)...
  Found 714 images for 2020
  Processing first 50 of 714 images...
    Processing image 1/50... (collected 0 fires so far)


c:\Users\Drewo\.conda\envs\fire_prediction\lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for MODIS/006/MOD14A1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by MODIS/061/MOD14A1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MOD14A1

  warnings.warn(warning, category=DeprecationWarning)
c:\Users\Drewo\.conda\envs\fire_prediction\lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for MODIS/006/MYD14A1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by MODIS/061/MYD14A1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MYD14A1

  warnings.warn(warning, category=DeprecationWarning)


    Processing image 11/50... (collected 2000 fires so far)
  ✓ Year 2020 complete: 4000 total fires collected
✓ Reached target of 4000 fires, stopping early

✓ Retrieved 4000 fire detections from GEE
✓ Sampled to 3000 detections with geographic diversity

✓ Saved raw fire detections to data\raw\fire_detections_gee.parquet
  Total detections: 3000
  Date range: 2020-01-01 00:00:00 to 2020-01-20 00:00:00


## Step 2.5: Generate Negative Samples (Non-Fire Locations)


In [ ]:
# ============================================================================
# GENERATE NEGATIVE SAMPLES (NON-FIRE LOCATIONS)
# ============================================================================

if GENERATE_NEGATIVE_SAMPLES and 'fire_data' in locals() and len(fire_data) > 0:
    print("="*80)
    print("GENERATING NEGATIVE SAMPLES (NON-FIRE LOCATIONS)")
    print("="*80)
    
    print(f"\nStrategy: {NEGATIVE_SAMPLE_STRATEGY}")
    print(f"Target: {N_NEGATIVE_SAMPLES} negative samples")
    
    # Get geographic bounds from fire data
    min_lat, max_lat = fire_data['LATITUDE'].min(), fire_data['LATITUDE'].max()
    min_lon, max_lon = fire_data['LONGITUDE'].min(), fire_data['LONGITUDE'].max()
    date_range = pd.date_range(GEE_FIRE_DATE_START, GEE_FIRE_DATE_END, freq='D')
    
    # Create sets for fast lookup
    fire_locations = set(zip(
        fire_data['LATITUDE'].round(3),  # Round to avoid exact matches
        fire_data['LONGITUDE'].round(3)
    ))
    fire_dates = set(fire_data['ACQ_DATE'].dt.date)
    
    negative_samples = []
    np.random.seed(RANDOM_STATE)
    
    if NEGATIVE_SAMPLE_STRATEGY == 'spatial_temporal':
        print("  - Sample locations near fire regions but at different times")
        print("  - Ensures similar geographic/terrain features but no fire")
        
        attempts = 0
        max_attempts = N_NEGATIVE_SAMPLES * 20  # Allow more attempts
        
        while len(negative_samples) < N_NEGATIVE_SAMPLES and attempts < max_attempts:
            attempts += 1
            
            # Sample a random location within fire region bounds
            lat = np.random.uniform(min_lat, max_lat)
            lon = np.random.uniform(min_lon, max_lon)
            
            # Sample a random date
            random_date = np.random.choice(date_range)
            
            # Check if this location+date combination has a fire
            location_key = (round(lat, 3), round(lon, 3))
            date_key = pd.Timestamp(random_date).date()
            
            # If no fire at this location+date, use it as negative sample
            if (location_key not in fire_locations) or (date_key not in fire_dates):
                negative_samples.append({
                    'LONGITUDE': lon,
                    'LATITUDE': lat,
                    'ACQ_DATE': random_date,
                    'BRIGHTNESS': 0,  # No fire
                    'FRP': 0,
                    'CONFIDENCE': 0,  # No fire = 0 confidence
                    'SATELLITE': 'NONE',
                    'VERSION': 'N/A',
                    'DAYNIGHT': 'D',
                    'is_fire': 0  # Label as negative sample (no fire)
                })
            
            if attempts % 1000 == 0:
                print(f"    Attempts: {attempts}, Generated: {len(negative_samples)}/{N_NEGATIVE_SAMPLES}")
        
        print(f"✓ Generated {len(negative_samples)} negative samples ({attempts} attempts)")
        
    elif NEGATIVE_SAMPLE_STRATEGY == 'grid':
        print("  - Create grid over fire region")
        print("  - Sample grid cells without fires")
        
        # Create grid
        n_grid_lat = 20
        n_grid_lon = 20
        lat_step = (max_lat - min_lat) / n_grid_lat
        lon_step = (max_lon - min_lon) / n_grid_lon
        
        for i in range(n_grid_lat):
            for j in range(n_grid_lon):
                if len(negative_samples) >= N_NEGATIVE_SAMPLES:
                    break
                
                # Grid cell center with small random offset
                lat = min_lat + i * lat_step + lat_step/2 + np.random.uniform(-lat_step/4, lat_step/4)
                lon = min_lon + j * lon_step + lon_step/2 + np.random.uniform(-lon_step/4, lon_step/4)
                
                # Check if this grid cell has fires
                nearby_fires = fire_data[
                    (fire_data['LATITUDE'].between(lat - lat_step/2, lat + lat_step/2)) &
                    (fire_data['LONGITUDE'].between(lon - lon_step/2, lon + lon_step/2))
                ]
                
                if len(nearby_fires) == 0:
                    # No fires in this grid cell - use as negative sample
                    random_date = np.random.choice(date_range)
                    negative_samples.append({
                        'LONGITUDE': lon,
                        'LATITUDE': lat,
                        'ACQ_DATE': random_date,
                        'BRIGHTNESS': 0,
                        'FRP': 0,
                        'CONFIDENCE': 0,
                        'SATELLITE': 'NONE',
                        'VERSION': 'N/A',
                        'DAYNIGHT': 'D',
                        'is_fire': 0
                    })
            
            if len(negative_samples) >= N_NEGATIVE_SAMPLES:
                break
        
        print(f"✓ Generated {len(negative_samples)} negative samples from grid")
        
    else:  # 'random'
        print("  - Random locations and dates in fire region")
        
        for _ in range(N_NEGATIVE_SAMPLES):
            lat = np.random.uniform(min_lat, max_lat)
            lon = np.random.uniform(min_lon, max_lon)
            random_date = np.random.choice(date_range)
            
            negative_samples.append({
                'LONGITUDE': lon,
                'LATITUDE': lat,
                'ACQ_DATE': random_date,
                'BRIGHTNESS': 0,
                'FRP': 0,
                'CONFIDENCE': 0,
                'SATELLITE': 'NONE',
                'VERSION': 'N/A',
                'DAYNIGHT': 'D',
                'is_fire': 0
            })
        
        print(f"✓ Generated {len(negative_samples)} random negative samples")
    
    # Convert to DataFrame
    if len(negative_samples) > 0:
        negative_df = pd.DataFrame(negative_samples)
        negative_df['ACQ_DATE'] = pd.to_datetime(negative_df['ACQ_DATE'])
        
        # Combine positive (fire) and negative (non-fire) samples
        print(f"\n📊 Dataset Summary:")
        print(f"  Positive samples (fires): {len(fire_data)}")
        print(f"  Negative samples (no fires): {len(negative_df)}")
        print(f"  Total samples: {len(fire_data) + len(negative_df)}")
        
        # Combine datasets
        combined_fire_data = pd.concat([
            fire_data,  # Already has is_fire=1
            negative_df  # Has is_fire=0
        ], ignore_index=True)
        
        # Shuffle the combined dataset
        combined_fire_data = combined_fire_data.sample(
            frac=1, 
            random_state=RANDOM_STATE
        ).reset_index(drop=True)
        
        print(f"\n✓ Combined dataset: {len(combined_fire_data)} samples")
        print(f"  Class distribution:")
        print(f"    Fire (1): {(combined_fire_data['is_fire'] == 1).sum()}")
        print(f"    No Fire (0): {(combined_fire_data['is_fire'] == 0).sum()}")
        
        # ============================================================================
        # GEOGRAPHIC VERIFICATION
        # ============================================================================
        print(f"\n{'='*80}")
        print("GEOGRAPHIC VERIFICATION (Positive vs Negative Samples)")
        print(f"{'='*80}")
        
        # Get bounds for each class
        fire_samples = combined_fire_data[combined_fire_data['is_fire'] == 1]
        no_fire_samples = combined_fire_data[combined_fire_data['is_fire'] == 0]
        
        fire_bounds = {
            'min_lat': fire_samples['LATITUDE'].min(),
            'max_lat': fire_samples['LATITUDE'].max(),
            'min_lon': fire_samples['LONGITUDE'].min(),
            'max_lon': fire_samples['LONGITUDE'].max()
        }
        
        no_fire_bounds = {
            'min_lat': no_fire_samples['LATITUDE'].min(),
            'max_lat': no_fire_samples['LATITUDE'].max(),
            'min_lon': no_fire_samples['LONGITUDE'].min(),
            'max_lon': no_fire_samples['LONGITUDE'].max()
        }
        
        print(f"\n📍 Geographic Bounds:")
        print(f"  Fire samples:")
        print(f"    Latitude:  [{fire_bounds['min_lat']:.4f}, {fire_bounds['max_lat']:.4f}]")
        print(f"    Longitude: [{fire_bounds['min_lon']:.4f}, {fire_bounds['max_lon']:.4f}]")
        print(f"  No-fire samples:")
        print(f"    Latitude:  [{no_fire_bounds['min_lat']:.4f}, {no_fire_bounds['max_lat']:.4f}]")
        print(f"    Longitude: [{no_fire_bounds['min_lon']:.4f}, {no_fire_bounds['max_lon']:.4f}]")
        
        # Check if negative samples are within fire bounds
        lat_overlap = (no_fire_bounds['min_lat'] >= fire_bounds['min_lat']) and \
                     (no_fire_bounds['max_lat'] <= fire_bounds['max_lat'])
        lon_overlap = (no_fire_bounds['min_lon'] >= fire_bounds['min_lon']) and \
                     (no_fire_bounds['max_lon'] <= fire_bounds['max_lon'])
        
        # Calculate overlap percentage
        lat_range_fire = fire_bounds['max_lat'] - fire_bounds['min_lat']
        lon_range_fire = fire_bounds['max_lon'] - fire_bounds['min_lon']
        lat_range_no_fire = no_fire_bounds['max_lat'] - no_fire_bounds['min_lat']
        lon_range_no_fire = no_fire_bounds['max_lon'] - no_fire_bounds['min_lon']
        
        lat_overlap_pct = (lat_range_no_fire / lat_range_fire * 100) if lat_range_fire > 0 else 0
        lon_overlap_pct = (lon_range_no_fire / lon_range_fire * 100) if lon_range_fire > 0 else 0
        
        print(f"\n✓ Geographic Overlap Analysis:")
        print(f"  Negative samples within fire bounds: {'✓ YES' if (lat_overlap and lon_overlap) else '⚠ PARTIAL'}")
        print(f"  Latitude overlap: {lat_overlap_pct:.1f}%")
        print(f"  Longitude overlap: {lon_overlap_pct:.1f}%")
        
        # Calculate centroid distances
        fire_centroid_lat = fire_samples['LATITUDE'].mean()
        fire_centroid_lon = fire_samples['LONGITUDE'].mean()
        no_fire_centroid_lat = no_fire_samples['LATITUDE'].mean()
        no_fire_centroid_lon = no_fire_samples['LONGITUDE'].mean()
        
        # Approximate distance in km (Haversine approximation)
        lat_diff = abs(fire_centroid_lat - no_fire_centroid_lat)
        lon_diff = abs(fire_centroid_lon - no_fire_centroid_lon)
        centroid_distance_km = np.sqrt(lat_diff**2 + lon_diff**2) * 111.0  # Rough conversion
        
        print(f"\n  Centroid locations:")
        print(f"    Fire:     ({fire_centroid_lat:.4f}, {fire_centroid_lon:.4f})")
        print(f"    No-fire:  ({no_fire_centroid_lat:.4f}, {no_fire_centroid_lon:.4f})")
        print(f"    Distance: {centroid_distance_km:.2f} km")
        
        if lat_overlap and lon_overlap:
            print(f"\n✅ Geographic similarity: EXCELLENT")
            print(f"   Negative samples are within the same geographic region as positive samples.")
            print(f"   This ensures the model learns from weather/temporal conditions, not just geography.")
        else:
            print(f"\n⚠ Geographic similarity: PARTIAL")
            print(f"   Some negative samples extend beyond fire region bounds.")
            print(f"   Consider adjusting NEGATIVE_SAMPLE_STRATEGY or geographic bounds.")
        
        print(f"{'='*80}\n")
        
        # Update fire_data variable for subsequent cells
        fire_data = combined_fire_data
        
        # Save combined dataset
        output_path = RAW_DIR / 'fire_detections_with_negatives_gee.parquet'
        fire_data.to_parquet(output_path, index=False)
        print(f"✓ Saved combined dataset to {output_path}")
    else:
        print(f"\n⚠ WARNING: Could not generate negative samples. Proceeding with fire data only.")
        print(f"  Consider adjusting NEGATIVE_SAMPLE_STRATEGY or increasing max_attempts.")
    
    print("="*80)
else:
    if 'fire_data' not in locals() or len(fire_data) == 0:
        print("⚠ Skipping negative sample generation: No fire data available")
    else:
        print("⏭ Skipping negative sample generation (GENERATE_NEGATIVE_SAMPLES=False)")


GENERATING NEGATIVE SAMPLES (NON-FIRE LOCATIONS)

Strategy: spatial_temporal
Target: 3000 negative samples
  - Sample locations near fire regions but at different times
  - Ensures similar geographic/terrain features but no fire
    Attempts: 1000, Generated: 1000/3000
    Attempts: 2000, Generated: 2000/3000
    Attempts: 3000, Generated: 3000/3000
✓ Generated 3000 negative samples (3000 attempts)

📊 Dataset Summary:
  Positive samples (fires): 3000
  Negative samples (no fires): 3000
  Total samples: 6000

✓ Combined dataset: 6000 samples
  Class distribution:
    Fire (1): 3000
    No Fire (0): 3000

✓ Saved combined dataset to data\raw\fire_detections_with_negatives_gee.parquet


In [6]:
# ============================================================================
# FETCH ERA5 WEATHER DATA FROM GEE
# ============================================================================

if FETCH_WEATHER and 'fire_data' in locals():
    print("="*80)
    print("FETCHING ERA5 WEATHER DATA FROM GOOGLE EARTH ENGINE")
    print("="*80)
    
    print(f"\nConfiguration:")
    print(f"  Product: {GEE_WEATHER_PRODUCT}")
    print(f"  Variables: {len(GEE_WEATHER_BANDS)}")
    print(f"  Lookback: {WEATHER_LOOKBACK_DAYS} days")
    print(f"  Total locations: {len(fire_data)}")
    
    # Load ERA5-Land daily aggregated data
    era5 = ee.ImageCollection(GEE_WEATHER_PRODUCT)
    
    weather_features = []
    
    print(f"\nProcessing weather data...")
    
    for idx, row in fire_data.iterrows():
        if idx % 100 == 0:
            print(f"  Processing {idx}/{len(fire_data)}...")
        
        lat = row['LATITUDE']
        lon = row['LONGITUDE']
        fire_date = pd.to_datetime(row['ACQ_DATE'])
        
        # Calculate date range
        start_date = (fire_date - timedelta(days=WEATHER_LOOKBACK_DAYS)).strftime('%Y-%m-%d')
        end_date = fire_date.strftime('%Y-%m-%d')
        
        # Filter ERA5 for this location and time period
        weather = era5.filterDate(start_date, end_date) \
            .select(GEE_WEATHER_BANDS)
        
        # Define point
        point = ee.Geometry.Point([lon, lat])
        
        try:
            # Extract values at point
            weather_series = weather.getRegion(point, 11132).getInfo()  # 11km scale for ERA5-Land
            
            # Parse the results
            if len(weather_series) > 1:
                headers = weather_series[0]
                data_rows = weather_series[1:]
                
                # Convert to DataFrame for easy aggregation
                weather_df = pd.DataFrame(data_rows, columns=headers)
                
                # Calculate features
                feature_dict = {
                    'fire_index': idx,
                    'latitude': lat,
                    'longitude': lon
                }
                
                # Temperature features (convert K to C)
                if 'temperature_2m' in headers:
                    temps = weather_df['temperature_2m'].astype(float) - 273.15
                    feature_dict['temp_mean'] = temps.mean()
                    feature_dict['temp_max'] = temps.max()
                    feature_dict['temp_min'] = temps.min()
                    feature_dict['temp_range'] = temps.max() - temps.min()
                
                # Temperature max/min
                if 'temperature_2m_max' in headers:
                    feature_dict['temp_extreme_max'] = weather_df['temperature_2m_max'].astype(float).max() - 273.15
                if 'temperature_2m_min' in headers:
                    feature_dict['temp_extreme_min'] = weather_df['temperature_2m_min'].astype(float).min() - 273.15
                
                # Humidity (from dewpoint)
                if 'dewpoint_temperature_2m' in headers and 'temperature_2m' in headers:
                    # Calculate relative humidity from dewpoint and temperature
                    T = weather_df['temperature_2m'].astype(float)
                    Td = weather_df['dewpoint_temperature_2m'].astype(float)
                    RH = 100 * (np.exp((17.625 * Td) / (243.04 + Td)) / np.exp((17.625 * T) / (243.04 + T)))
                    feature_dict['humidity_mean'] = RH.mean()
                    feature_dict['humidity_min'] = RH.min()
                
                # Precipitation (convert m to mm)
                if 'total_precipitation_sum' in headers:
                    precip = weather_df['total_precipitation_sum'].astype(float) * 1000
                    feature_dict['precip_total'] = precip.sum()
                    feature_dict['precip_mean'] = precip.mean()
                    feature_dict['precip_max'] = precip.max()
                    feature_dict['days_no_rain'] = (precip < 0.1).sum()
                
                # Wind (combine U and V components)
                if 'u_component_of_wind_10m' in headers and 'v_component_of_wind_10m' in headers:
                    u = weather_df['u_component_of_wind_10m'].astype(float)
                    v = weather_df['v_component_of_wind_10m'].astype(float)
                    wind_speed = np.sqrt(u**2 + v**2)
                    feature_dict['wind_speed_mean'] = wind_speed.mean()
                    feature_dict['wind_speed_max'] = wind_speed.max()
                
                # Pressure
                if 'surface_pressure' in headers:
                    pressure = weather_df['surface_pressure'].astype(float) / 100  # Pa to hPa
                    feature_dict['pressure_mean'] = pressure.mean()
                
                weather_features.append(feature_dict)
                
        except Exception as e:
            if idx % 100 == 0:
                print(f"    Warning: Failed to fetch weather for fire {idx}: {str(e)[:50]}")
            continue
    
    print(f"\n✓ Retrieved weather data for {len(weather_features)}/{len(fire_data)} locations")
    
    # Convert to DataFrame
    weather_df_all = pd.DataFrame(weather_features)
    
    if len(weather_df_all) > 0:
        # Save weather features
        output_path = PROCESSED_DIR / 'weather_features_era5.parquet'
        weather_df_all.to_parquet(output_path, index=False)
        print(f"✓ Saved to {output_path}")
    else:
        print(f"✗ No weather features extracted")
    
    print("="*80)
    
else:
    if 'fire_data' not in locals():
        print("No fire data available. Run fire detection first.")
    else:
        print("⏭ Skipping weather features (FETCH_WEATHER=False)")


FETCHING ERA5 WEATHER DATA FROM GOOGLE EARTH ENGINE

Configuration:
  Product: ECMWF/ERA5_LAND/DAILY_AGGR
  Variables: 8
  Lookback: 14 days
  Total locations: 6000

Processing weather data...
  Processing 0/6000...
  Processing 100/6000...
  Processing 200/6000...
  Processing 300/6000...
  Processing 400/6000...
  Processing 500/6000...
  Processing 600/6000...
  Processing 700/6000...
  Processing 800/6000...
  Processing 900/6000...
  Processing 1000/6000...
  Processing 1100/6000...
  Processing 1200/6000...
  Processing 1300/6000...
  Processing 1400/6000...
  Processing 1500/6000...
  Processing 1600/6000...
  Processing 1700/6000...
  Processing 1800/6000...
  Processing 1900/6000...
  Processing 2000/6000...
  Processing 2100/6000...
  Processing 2200/6000...
  Processing 2300/6000...
  Processing 2400/6000...
  Processing 2500/6000...
  Processing 2600/6000...
  Processing 2700/6000...
  Processing 2800/6000...
  Processing 2900/6000...
  Processing 3000/6000...
  Processing 

## Step 4: Fetch Terrain Features from GEE


In [7]:
# ============================================================================
# FETCH TERRAIN FEATURES FROM GEE
# ============================================================================

if FETCH_TERRAIN and 'fire_data' in locals():
    try:
        from data_ingest.google_gee.get_terrain_features import get_terrain_features
        
        print("="*80)
        print("FETCHING TERRAIN FEATURES FROM GOOGLE EARTH ENGINE")
        print("="*80)
        print("Using SRTM DEM via GEE (no rate limits, faster processing)")
        print(f"Processing {len(fire_data)} locations...")
        print("="*80)
        
        terrain_features = []
        
        for idx, row in fire_data.iterrows():
            if idx % 100 == 0:
                print(f"  Processing {idx+1}/{len(fire_data)}...")
            
            try:
                # Fetch terrain features from GEE
                terrain = get_terrain_features(
                    latitude=row['LATITUDE'],
                    longitude=row['LONGITUDE'],
                    scale_meters=TERRAIN_SCALE_METERS,
                    buffer_km=TERRAIN_BUFFER_KM,
                    fire_date=row['ACQ_DATE'].strftime('%Y-%m-%d') if 'ACQ_DATE' in row else None
                )
                
                terrain['fire_index'] = idx
                terrain_features.append(terrain)
                
            except Exception as e:
                if idx % 100 == 0:
                    print(f"  Warning: Error processing fire {idx}: {e}")
                # Add NaN values for this location
                terrain_features.append({
                    'fire_index': idx,
                    'latitude': row['LATITUDE'],
                    'longitude': row['LONGITUDE'],
                    'elevation': np.nan,
                    'elevation_std': np.nan,
                    'slope': np.nan,
                    'slope_max': np.nan,
                    'ruggedness': np.nan,
                    'curvature': np.nan,
                    'canyons': np.nan
                })
                continue
        
        # Create DataFrame
        terrain_df = pd.DataFrame(terrain_features)
        
        if len(terrain_df) > 0:
            # Save terrain features
            output_path = PROCESSED_DIR / 'terrain_features.parquet'
            terrain_df.to_parquet(output_path, index=False)
            print(f"\n✓ Saved terrain features for {len(terrain_df)} locations to {output_path}")
        else:
            print("✗ No terrain features were successfully fetched")
        
    except ImportError as e:
        print(f"✗ Error importing GEE terrain module: {e}")
        print("  Make sure data_ingest.google_gee.get_terrain_features is available")
    except Exception as e:
        print(f"✗ Error fetching terrain features from GEE: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⏭ Skipping terrain features (FETCH_TERRAIN=False)")


FETCHING TERRAIN FEATURES FROM GOOGLE EARTH ENGINE
Using SRTM DEM via GEE (no rate limits, faster processing)
Processing 6000 locations...
  Processing 1/6000...
  Processing 101/6000...
  Processing 201/6000...
  Processing 301/6000...
  Processing 401/6000...
  Processing 501/6000...
  Processing 601/6000...
  Processing 701/6000...
  Processing 801/6000...
  Processing 901/6000...


KeyboardInterrupt: 

## Step 5: Fetch Geospatial Features (Water Distance & Vegetation) from GEE


In [ ]:
# ============================================================================
# FETCH GEOSPATIAL FEATURES FROM GEE (Water Distance & Vegetation)
# ============================================================================

if (FETCH_WATER_DISTANCE or FETCH_VEGETATION) and 'fire_data' in locals():
    try:
        # Import GEE modules
        from data_ingest.google_gee.get_water_distance import get_distance_to_water
        from data_ingest.google_gee.get_vegetation_features import (
            get_forest_types, 
            get_fuel_layers
        )
        
        print("="*80)
        print("FETCHING GEOSPATIAL FEATURES FROM GOOGLE EARTH ENGINE")
        print("="*80)
        print(f"Processing {len(fire_data)} locations...")
        print("="*80)
        
        geospatial_features = []
        
        for idx, row in fire_data.iterrows():
            if idx % 100 == 0:
                print(f"  Processing {idx+1}/{len(fire_data)}...")
            
            fire_date = pd.to_datetime(row['ACQ_DATE'])
            fire_date_str = fire_date.strftime('%Y-%m-%d')
            
            geospatial_feature = {
                'fire_index': idx,
                'latitude': row['LATITUDE'],
                'longitude': row['LONGITUDE'],
                'fire_date': fire_date_str
            }
            
            # Fetch water distance
            if FETCH_WATER_DISTANCE:
                try:
                    distance = get_distance_to_water(
                        latitude=row['LATITUDE'],
                        longitude=row['LONGITUDE'],
                        max_search_radius_km=50.0,
                        fire_date=fire_date_str
                    )
                    geospatial_feature['distance_to_water_meters'] = distance
                except Exception as e:
                    print(f"    Warning: Error fetching water distance for fire {idx}: {e}")
                    geospatial_feature['distance_to_water_meters'] = np.nan
            
            # Fetch vegetation features
            if FETCH_VEGETATION:
                try:
                    # Get forest types
                    forest_types = get_forest_types(
                        latitude=row['LATITUDE'],
                        longitude=row['LONGITUDE'],
                        fire_date=fire_date_str
                    )
                    
                    # Add forest type features
                    for key, value in forest_types.items():
                        if key not in ['latitude', 'longitude', 'fire_date', 'data_source']:
                            geospatial_feature[f'forest_{key}'] = value
                    
                    # Get fuel layers
                    fuel_layers = get_fuel_layers(
                        latitude=row['LATITUDE'],
                        longitude=row['LONGITUDE'],
                        fire_date=fire_date_str
                    )
                    
                    # Add fuel layer features
                    for key, value in fuel_layers.items():
                        if key not in ['latitude', 'longitude', 'fire_date', 'data_source']:
                            geospatial_feature[f'fuel_{key}'] = value
                            
                except Exception as e:
                    print(f"    Warning: Error fetching vegetation for fire {idx}: {e}")
            
            geospatial_features.append(geospatial_feature)
        
        # Create DataFrame
        geospatial_df = pd.DataFrame(geospatial_features)
        
        if len(geospatial_df) > 0:
            # Save geospatial features
            output_path = PROCESSED_DIR / 'geospatial_features.parquet'
            geospatial_df.to_parquet(output_path, index=False)
            print(f"\n✓ Saved geospatial features for {len(geospatial_df)} locations to {output_path}")
        else:
            print("✗ No geospatial features were successfully fetched")
            
    except ImportError as e:
        print(f"✗ Error importing GEE modules: {e}")
        print("  Make sure data_ingest.google_gee modules are available")
    except Exception as e:
        print(f"✗ Error fetching geospatial features: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⏭ Skipping geospatial features (FETCH_WATER_DISTANCE=False and FETCH_VEGETATION=False)")


## Step 6: Combine All Features


In [ ]:
# ============================================================================
# COMBINE ALL FEATURES
# ============================================================================

print("="*80)
print("COMBINING ALL FEATURES")
print("="*80)

# Start with fire data
combined_df = fire_data.copy()
combined_df['fire_index'] = combined_df.index

# Merge weather features
if FETCH_WEATHER:
    weather_loaded = False
    if 'weather_df_all' in locals() and weather_df_all is not None:
        try:
            weather_merge = weather_df_all.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
            combined_df = combined_df.merge(weather_merge, on='fire_index', how='left')
            print(f"✓ Merged weather features from memory ({len(weather_df_all)} records)")
            weather_loaded = True
        except Exception as e:
            print(f"⚠ Error merging weather from memory: {e}")
    
    # Try loading from saved parquet file
    if not weather_loaded:
        weather_parquet = PROCESSED_DIR / 'weather_features_era5.parquet'
        if weather_parquet.exists():
            try:
                weather_df_loaded = pd.read_parquet(weather_parquet)
                weather_merge = weather_df_loaded.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
                combined_df = combined_df.merge(weather_merge, on='fire_index', how='left')
                print(f"✓ Merged weather features from parquet ({len(weather_df_loaded)} records)")
            except Exception as e:
                print(f"✗ Error loading weather parquet: {e}")
        else:
            print(f"⚠ Weather parquet not found: {weather_parquet}")

# Merge terrain features
if FETCH_TERRAIN:
    terrain_loaded = False
    if 'terrain_df' in locals() and terrain_df is not None:
        try:
            terrain_merge = terrain_df.drop(columns=['latitude', 'longitude'], errors='ignore')
            combined_df = combined_df.merge(terrain_merge, on='fire_index', how='left')
            print(f"✓ Merged terrain features from memory ({len(terrain_df)} records)")
            terrain_loaded = True
        except Exception as e:
            print(f"⚠ Error merging terrain from memory: {e}")
    
    # Try loading from saved parquet file
    if not terrain_loaded:
        terrain_parquet = PROCESSED_DIR / 'terrain_features.parquet'
        if terrain_parquet.exists():
            try:
                terrain_df_loaded = pd.read_parquet(terrain_parquet)
                terrain_merge = terrain_df_loaded.drop(columns=['latitude', 'longitude'], errors='ignore')
                combined_df = combined_df.merge(terrain_merge, on='fire_index', how='left')
                print(f"✓ Merged terrain features from parquet ({len(terrain_df_loaded)} records)")
            except Exception as e:
                print(f"✗ Error loading terrain parquet: {e}")
        else:
            print(f"⚠ Terrain parquet not found: {terrain_parquet}")

# Merge geospatial features
if FETCH_WATER_DISTANCE or FETCH_VEGETATION:
    geospatial_loaded = False
    if 'geospatial_df' in locals() and geospatial_df is not None:
        try:
            geospatial_merge = geospatial_df.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
            combined_df = combined_df.merge(geospatial_merge, on='fire_index', how='left')
            print(f"✓ Merged geospatial features from memory ({len(geospatial_df)} records)")
            geospatial_loaded = True
        except Exception as e:
            print(f"⚠ Error merging geospatial from memory: {e}")
    
    # Try loading from saved parquet file
    if not geospatial_loaded:
        geospatial_parquet = PROCESSED_DIR / 'geospatial_features.parquet'
        if geospatial_parquet.exists():
            try:
                geospatial_df_loaded = pd.read_parquet(geospatial_parquet)
                geospatial_merge = geospatial_df_loaded.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
                combined_df = combined_df.merge(geospatial_merge, on='fire_index', how='left')
                print(f"✓ Merged geospatial features from parquet ({len(geospatial_df_loaded)} records)")
            except Exception as e:
                print(f"✗ Error loading geospatial parquet: {e}")
        else:
            print(f"⚠ Geospatial parquet not found: {geospatial_parquet}")

# Save combined dataset
output_path = PROCESSED_DIR / 'ml_ready_gee.parquet'
combined_df.to_parquet(output_path, index=False)
print(f"\n✓ Saved combined features dataset ({len(combined_df)} rows, {len(combined_df.columns)} columns)")
print(f"  Output: {output_path}")

print(f"\nColumn categories:")
print(f"  - Fire detection: {len([c for c in combined_df.columns if c in ['LATITUDE', 'LONGITUDE', 'BRIGHTNESS', 'CONFIDENCE', 'FRP']])}")
print(f"  - Weather: {len([c for c in combined_df.columns if any(x in c.lower() for x in ['temp', 'humidity', 'precip', 'wind', 'pressure'])])}")
print(f"  - Terrain: {len([c for c in combined_df.columns if c in ['elevation', 'slope', 'ruggedness', 'curvature', 'canyons']])}")
print(f"  - Geospatial: {len([c for c in combined_df.columns if 'distance_to_water' in c.lower() or 'forest' in c.lower() or 'fuel' in c.lower()])}")

print(f"\n✓ Data ingestion complete!")
print(f"  Combined dataset saved to: {output_path}")
print("="*80)
